# 09 - Final Analysis, Processed Datasets and Insights
**Project:** Air Quality & Pollution Intelligence - Data Mining and Business Intelligence

**Objective:** Run the complete pipeline once more from a single entry point, export every processed dataset for Power BI, and turn the computed results into the insight list.

**How to read this notebook:** every number printed below is produced by the
code in this notebook from `data/raw/Air_quality_data.csv`. Column names are
discovered at runtime through `src/config.py`, so nothing is assumed.


In [ ]:
"""Environment bootstrap: make src/ importable and pin the working directory."""
import sys, os, warnings
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 90)
%matplotlib inline
print("project root:", PROJECT_ROOT)

## 1. Run the whole pipeline from the shared code

`src/pipeline.py` is the same code the notebooks above use, so the exported files
cannot drift away from what was demonstrated.

In [ ]:
import json
import pipeline
import config as C
from IPython.display import display

results = pipeline.run(verbose=True)
print("runtime (seconds):", results["runtime_seconds"])

## 2. Environment recorded for reproducibility

In [ ]:
display(pd.DataFrame(list(results["environment"].items()),
                     columns=["Component", "Version / setting"]))

## 3. Dataset facts used in the report

In [ ]:
ds = results["dataset"]
display(pd.DataFrame(list(ds.items()), columns=["Item", "Value"]))

## 4. Every processed dataset that Power BI will import

In [ ]:
files = sorted(C.PROCESSED_DIR.glob("*.csv"))
table = pd.DataFrame({
    "file": [f.name for f in files],
    "rows": [sum(1 for _ in open(f, encoding="utf-8")) - 1 for f in files],
    "columns": [pd.read_csv(f, nrows=1).shape[1] for f in files]})
display(table)
print("primary BI inputs:")
for k, v in results["exports"].items():
    print(f"  {k:<22}{Path(v).name}")

## 5. Insights generated from the computed results

Each insight is assembled by `src/reporting.py` from numbers that exist in the
artefacts listed above, and carries its own evidence column and source file.

In [ ]:
ins = pd.DataFrame(results["insights"])
display(ins[["Category", "Insight", "Evidence Strength"]])
print("full table with evidence:")
display(ins[["Insight", "Quantitative Evidence", "Derived From"]])

## 6. Key result tables in one place

In [ ]:
display(pd.DataFrame(results["classification"]["comparison"])[
    ["Model", "Accuracy_%", "F1_macro", "Lift_over_Baseline_pp"]].head())
display(pd.DataFrame(results["clustering"]["choice"], index=["value"]).T)
display(pd.DataFrame([results["anomaly"]["stats"]]))
display(pd.DataFrame([results["predictability"]]))
display(pd.DataFrame([results["trend_test"]]))

## 7. Final quality check

Every line below is evaluated from the artefacts, not asserted. Two checks deserve
an explanation:

* **Measured columns** (city, date, the nine pollutant concentrations, AQI) must
  contain no blank cells at all.
* **Derived columns** are *allowed* to be blank where the derivation is undefined:
  a ratio whose denominator is zero, and the first day of each city's lag / rolling
  window, which has no previous observation to compare with. Those blanks are
  documented, not silently filled.

In [ ]:
import data_utils as U

cleaned = pd.read_csv(C.CLEANED_CSV)
measured = ([C.CITY_COL, "Date"]
            + C.pollutant_columns(cleaned.columns)
            + [c for c in (C.AQI_COL, C.TARGET_COL) if c in cleaned.columns])
blank_cols = set(cleaned.columns[cleaned.isna().any()])
allowed_blank = {"PM25_PM10_Ratio", "NO2_NOx_Ratio", "O3_PM10_Ratio",
                 "AQI_Rolling_7", "AQI_Rolling_30", "AQI_Change_1d",
                 "Cluster_Description", "City_Pollutant_Percentile"}
# Rebuild the category from the AQI value with the single band table in
# src/config.py (through U.band_from_aqi), then count how often it agrees with
# the supplied label - this is the leakage test and the scale test at once.
expected_bucket = U.band_from_aqi(cleaned[C.AQI_COL])
bucket_agreement = float((expected_bucket.astype(str)
                          == cleaned[C.TARGET_COL].astype(str)).mean())
checks = {
 "raw dataset loads": C.RAW_FILE.exists(),
 "cleaned dataset exported": C.CLEANED_CSV.exists(),
 "city summary exported": C.CITY_SUMMARY_CSV.exists(),
 "monthly summary exported": C.MONTHLY_SUMMARY_CSV.exists(),
 "cluster results exported": C.CLUSTER_RESULTS_CSV.exists(),
 "anomaly results exported": C.ANOMALY_RESULTS_CSV.exists(),
 "classification results exported": C.CLASSIFICATION_CSV.exists(),
 "model comparison exported": C.MODEL_COMPARISON_CSV.exists(),
 "insights exported": C.INSIGHTS_CSV.exists(),
 "date / bucket / city dimensions exported":
     C.DIM_DATE_CSV.exists() and C.DIM_BUCKET_CSV.exists() and C.DIM_CITY_CSV.exists(),
 "results.json written": (C.PROCESSED_DIR / "results.json").exists(),
 "no blank cells in the measured columns":
     int(cleaned[measured].isna().sum().sum()) == 0,
 "blank cells confined to documented derived columns": bool(blank_cols <= allowed_blank),
 "one record per city-day":
     not cleaned.duplicated([C.CITY_COL, "Date"]).any(),
 "AQI_Bucket is reproducible from AQI (leakage confirmed)": bucket_agreement == 1.0,
}
display(pd.DataFrame(list(checks.items()), columns=["Check", "Passed"]))
print("all checks passed:", all(checks.values()))
print()
print("blank cells by column (only derived columns may appear):")
blanks = cleaned.isna().sum()
display(blanks[blanks > 0].rename("Blank cells").to_frame())
print(f"AQI_Bucket agreement with the band table: {100 * bucket_agreement:.2f}%")
print("disagreements:", int((expected_bucket.astype(str)
                            != cleaned[C.TARGET_COL].astype(str)).sum()))

## 8. Star-schema tables for Power BI

`FACT_AirQuality` is `air_quality_cleaned.csv`; the three dimension tables below
are what a clean model needs (see `dashboard/POWER_BI_BUILD_GUIDE.md`).

In [ ]:
print("DIM_Date rows:", f"{pd.read_csv(C.DIM_DATE_CSV).shape[0]:,}")
display(pd.read_csv(C.DIM_BUCKET_CSV))
display(pd.read_csv(C.DIM_CITY_CSV))

## 9. Handoff to Power BI

The CSVs in `data/processed/` are the only inputs the dashboard needs. The star
schema, the DAX measures and the page-by-page build instructions live in
`dashboard/` - see `dashboard/POWER_BI_BUILD_GUIDE.md`.

In [ ]:
print("Import these into Power BI Desktop:")
for name in ["air_quality_cleaned.csv", "dim_date.csv", "dim_city.csv",
             "dim_bucket.csv", "city_summary.csv", "monthly_summary.csv",
             "cluster_results.csv", "cluster_profile.csv",
             "anomaly_results.csv", "aqi_bucket_distribution.csv",
             "model_comparison.csv", "classification_per_class.csv",
             "correlation_with_aqi.csv", "insights.csv",
             "data_quality_report.csv", "k_selection_table.csv"]:
    p = C.PROCESSED_DIR / name
    print(f"  {'OK  ' if p.exists() else 'MISS'} {name}")

## 10. Methodology diagram (figure used in the report)

The chain below is the method this project actually followed, stage by stage, and
the italic line under each stage names the file that evidences it - so the
diagram can be checked against `data/processed/` rather than taken on trust.

In [ ]:
import figures as F
from IPython.display import Image

method_png = F.methodology_diagram()
print("written to:", Path(method_png).relative_to(C.PROJECT_ROOT))
Image(filename=method_png)